In [1]:
from pathlib import Path
from collections import Counter
from qonnx.core.modelwrapper import ModelWrapper

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
IM_DIR = BUILD_DIR / "intermediate_models"

models_to_check = [
    "step_streamline.onnx",
    "step_convert_to_hw.onnx",
    "step_create_dataflow_partition.onnx",
    "step_set_fifo_depths.onnx",
]

def summarize_model(path):
    print("\n" + "=" * 80)
    print(path)
    print("=" * 80)

    if not path.exists():
        print("[missing]")
        return

    model = ModelWrapper(str(path))
    nodes = list(model.graph.node)

    print("Number of nodes:", len(nodes))

    op_counter = Counter([n.op_type for n in nodes])
    domain_counter = Counter([n.domain for n in nodes])

    print("\nOp types:")
    for op, cnt in op_counter.most_common():
        print(f"  {op:40s} {cnt}")

    print("\nDomains:")
    for d, cnt in domain_counter.most_common():
        print(f"  {repr(d):40s} {cnt}")

    print("\nNode list:")
    for i, n in enumerate(nodes):
        print(f"{i:4d}: {n.name:50s} op={n.op_type:35s} domain={n.domain}")

for name in models_to_check:
    summarize_model(IM_DIR / name)


/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_streamline.onnx
Number of nodes: 64

Op types:
  Transpose                                18
  MultiThreshold                           13
  MatMul                                   9
  Mul                                      7
  Im2Col                                   7
  Pad                                      6
  Add                                      4

Domains:
  ''                                       44
  'qonnx.custom_op.general'                20

Node list:
   0: MultiThreshold_0                                   op=MultiThreshold                      domain=qonnx.custom_op.general
   1: Mul_0                                              op=Mul                                 domain=
   2: Transpose_0                                        op=Transpose                           domain=
   3: MatMul_0                                           op=MatMul             

In [3]:
from pathlib import Path
from collections import Counter
from qonnx.core.modelwrapper import ModelWrapper

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
IM_DIR = BUILD_DIR / "intermediate_models"

for name in [
    "step_streamline.onnx",
    "step_convert_to_hw.onnx",
    "step_create_dataflow_partition_patched.onnx",
    "step_set_fifo_depths.onnx",
]:
    path = IM_DIR / name
    print("\n" + "=" * 80)
    print(name)

    model = ModelWrapper(str(path))
    print(Counter([n.op_type for n in model.graph.node]))

    print("\nLast nodes:")
    for n in model.graph.node[-10:]:
        print(n.name, n.op_type, n.domain, "inputs=", list(n.input), "outputs=", list(n.output))


step_streamline.onnx
Counter({'Transpose': 18, 'MultiThreshold': 13, 'MatMul': 9, 'Mul': 7, 'Im2Col': 7, 'Pad': 6, 'Add': 4})

Last nodes:
MultiThreshold_11 MultiThreshold qonnx.custom_op.general inputs= ['MatMul_7_out0', 'MultiThreshold_11_param0'] outputs= ['MultiThreshold_11_out0']
Transpose_15 Transpose  inputs= ['MultiThreshold_11_out0'] outputs= ['Transpose_15_out0']
Add_2 Add  inputs= ['Transpose_15_out0', 'Transpose_12_out0'] outputs= ['Add_2_out0']
MultiThreshold_12 MultiThreshold qonnx.custom_op.general inputs= ['Add_2_out0', 'MultiThreshold_12_param0'] outputs= ['MultiThreshold_12_out0']
Transpose_16 Transpose  inputs= ['MultiThreshold_12_out0'] outputs= ['Transpose_16_out0']
Im2Col_6 Im2Col qonnx.custom_op.general inputs= ['Transpose_16_out0'] outputs= ['Im2Col_6_out0']
MatMul_8 MatMul  inputs= ['Im2Col_6_out0', 'MatMul_8_param0'] outputs= ['MatMul_8_out0']
Transpose_17 Transpose  inputs= ['MatMul_8_out0'] outputs= ['Transpose_17_out0']
Mul_6 Mul  inputs= ['Transpose_17_ou

In [5]:
from pathlib import Path
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
m = ModelWrapper(str(BUILD_DIR / "intermediate_models" / "step_streamline.onnx"))

for n in m.graph.node:
    if n.op_type in ["Mul", "Add"]:
        print("\n", n.name, n.op_type)
        print("inputs:", list(n.input))
        print("outputs:", list(n.output))
        for inp in n.input:
            init = m.get_initializer(inp)
            if init is not None:
                print("initializer", inp, "shape", init.shape, "value:")
                print(init)


 Mul_0 Mul
inputs: ['MultiThreshold_0_out0', 'Mul_0_param0']
outputs: ['Mul_0_out0']
initializer Mul_0_param0 shape () value:
0.125

 Mul_1 Mul
inputs: ['Transpose_3_out0', 'Mul_1_param0']
outputs: ['Mul_1_out0']
initializer Mul_1_param0 shape () value:
0.03125

 Add_0 Add
inputs: ['Transpose_5_out0', 'Transpose_2_out0']
outputs: ['Add_0_out0']

 Mul_2 Mul
inputs: ['MultiThreshold_4_out0', 'Mul_2_param0']
outputs: ['Mul_2_out0']
initializer Mul_2_param0 shape () value:
0.03125

 Mul_3 Mul
inputs: ['Transpose_7_out0', 'Mul_3_param0']
outputs: ['Mul_3_out0']
initializer Mul_3_param0 shape () value:
0.015625

 Add_1 Add
inputs: ['Transpose_9_out0', 'MultiThreshold_5_out0']
outputs: ['Add_1_out0']

 Mul_4 Mul
inputs: ['MultiThreshold_8_out0', 'Mul_4_param0']
outputs: ['Mul_4_out0']
initializer Mul_4_param0 shape () value:
0.0625

 Mul_5 Mul
inputs: ['Transpose_13_out0', 'Mul_5_param0']
outputs: ['Mul_5_out0']
initializer Mul_5_param0 shape () value:
0.015625

 Add_2 Add
inputs: ['Transpos

In [6]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
IM_DIR = BUILD_DIR / "intermediate_models"

for name in [
    "step_streamline.onnx",
    "step_convert_to_hw.onnx",
    "step_create_dataflow_partition.onnx",
    "step_set_fifo_depths.onnx",
]:
    path = IM_DIR / name
    print("\n" + "=" * 80)
    print(name)

    if not path.exists():
        print("missing")
        continue

    model = ModelWrapper(str(path))

    for tname in ["global_out"]:
        try:
            print(tname, "shape =", model.get_tensor_shape(tname))
            print(tname, "dtype =", model.get_tensor_datatype(tname))
        except Exception as e:
            print(tname, "not found:", e)


step_streamline.onnx
global_out shape = [1, 5, 1, 1]
global_out dtype = FLOAT32

step_convert_to_hw.onnx
global_out shape = [1, 5]
global_out dtype = FLOAT32

step_create_dataflow_partition.onnx
global_out shape = None
global_out dtype = FLOAT32

step_set_fifo_depths.onnx
global_out shape = [1, 1, 1, 5]
global_out dtype = INT32


In [7]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

# 改成你截图中那个 child model 的路径
MODEL_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx")

model = ModelWrapper(str(MODEL_PATH))

for n in model.graph.node:
    if "MVAU" in n.op_type:
        print("\nNode:", n.name, n.op_type)
        print("inputs :", list(n.input))
        print("outputs:", list(n.output))

        for t in list(n.input) + list(n.output):
            try:
                print(" ", t)
                print("   shape:", model.get_tensor_shape(t))
                print("   dtype:", model.get_tensor_datatype(t))
            except Exception as e:
                print("   cannot read:", e)

        try:
            inst = getCustomOp(n)
            for attr in [
                "inputDataType",
                "weightDataType",
                "outputDataType",
                "accDataType",
                "binaryXnorMode",
                "noActivation",
                "MW",
                "MH",
                "SIMD",
                "PE",
            ]:
                try:
                    print(" ", attr, "=", inst.get_nodeattr(attr))
                except:
                    pass
        except Exception as e:
            print("getCustomOp failed:", e)


Node: MVAU_hls_0 MVAU_hls
inputs : ['StreamingFIFO_rtl_2_out0', 'MVAU_hls_0_param0', 'MVAU_hls_0_param1']
outputs: ['MVAU_hls_0_out0']
  StreamingFIFO_rtl_2_out0
   shape: [1, 140, 1, 1]
   dtype: INT8
  MVAU_hls_0_param0
   shape: [1, 16]
   dtype: INT8
  MVAU_hls_0_param1
   shape: [16, 255]
   dtype: INT32
  MVAU_hls_0_out0
   shape: [1, 140, 1, 16]
   dtype: INT8
  inputDataType = INT8
  weightDataType = INT8
  outputDataType = INT8
  accDataType = INT15
  binaryXnorMode = 0
  noActivation = 0
  MW = 1
  MH = 16
  SIMD = 1
  PE = 1

Node: MVAU_hls_1 MVAU_hls
inputs : ['StreamingFIFO_rtl_5_out0', 'MVAU_hls_1_param0', 'MVAU_hls_1_param1']
outputs: ['MVAU_hls_1_out0']
  StreamingFIFO_rtl_5_out0
   shape: [1, 140, 1, 3]
   dtype: INT8
  MVAU_hls_1_param0
   shape: [3, 16]
   dtype: INT8
  MVAU_hls_1_param1
   shape: [16, 255]
   dtype: INT32
  MVAU_hls_1_out0
   shape: [1, 140, 1, 16]
   dtype: UINT8
  inputDataType = INT8
  weightDataType = INT8
  outputDataType = UINT8
  accDataType

In [9]:
import numpy as np
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx")
model = ModelWrapper(str(MODEL_PATH))

target_name = "MVAU_rtl_0"

for n in model.graph.node:
    if n.name == target_name:
        print("Found node:", n.name, n.op_type)
        print("inputs :", list(n.input))
        print("outputs:", list(n.output))

        w = model.get_initializer(n.input[1])
        w = np.asarray(w)

        print("\nFinal classifier weight:")
        print(" shape:", w.shape)
        print(" dtype:", w.dtype)
        print(" min/max:", w.min(), w.max())
        print(" nonzero:", np.count_nonzero(w), "/", w.size)

        vals, counts = np.unique(w, return_counts=True)
        print("\nunique values:")
        for v, c in zip(vals, counts):
            print(v, c)

        print("\nnonzero per output class:")
        for c in range(w.shape[1]):
            wc = w[:, c]
            print(
                "class", c,
                "nonzero", np.count_nonzero(wc), "/", wc.size,
                "min/max", wc.min(), wc.max(),
                "sum_abs", np.sum(np.abs(wc)),
                "sum", np.sum(wc),
            )

Found node: MVAU_rtl_0 MVAU_rtl
inputs : ['StreamingFIFO_rtl_47_out0', 'MVAU_rtl_0_param0']
outputs: ['MVAU_rtl_0_out0']

Final classifier weight:
 shape: (4480, 5)
 dtype: float32
 min/max: -70.0 68.0
 nonzero: 21882 / 22400

unique values:
-70.0 1
-69.0 2
-68.0 3
-67.0 5
-66.0 2
-65.0 4
-64.0 2
-63.0 4
-62.0 12
-61.0 11
-60.0 11
-59.0 14
-58.0 16
-57.0 24
-56.0 12
-55.0 21
-54.0 26
-53.0 30
-52.0 31
-51.0 37
-50.0 30
-49.0 38
-48.0 40
-47.0 56
-46.0 59
-45.0 59
-44.0 70
-43.0 84
-42.0 87
-41.0 90
-40.0 90
-39.0 114
-38.0 119
-37.0 109
-36.0 129
-35.0 154
-34.0 167
-33.0 147
-32.0 170
-31.0 176
-30.0 186
-29.0 197
-28.0 224
-27.0 218
-26.0 226
-25.0 241
-24.0 262
-23.0 261
-22.0 280
-21.0 271
-20.0 298
-19.0 322
-18.0 327
-17.0 343
-16.0 363
-15.0 368
-14.0 365
-13.0 362
-12.0 415
-11.0 408
-10.0 459
-9.0 439
-8.0 399
-7.0 409
-6.0 470
-5.0 553
-4.0 498
-3.0 414
-2.0 455
-1.0 440
0.0 518
1.0 454
2.0 433
3.0 362
4.0 384
5.0 437
6.0 393
7.0 291
8.0 318
9.0 345
10.0 306
11.0 290
12.0 302

In [1]:
import os
os.environ["SANITIZE_QUANT_TENSORS"] = "0"

import numpy as np
from pathlib import Path

from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe


BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
MODEL_PATH = BUILD_DIR / "intermediate_models" / "step_streamline.onnx"

TEST_TXT = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt")

# 改成你板上正在用的 inputscale
INPUT_SCALE = 1.0

model = ModelWrapper(str(MODEL_PATH))
model.set_metadata_prop("exec_mode", "")

data = np.loadtxt(TEST_TXT, dtype=np.float32)

label_raw = int(data[0, 0])
label = label_raw - 1 if label_raw in [1, 2, 3, 4, 5] else label_raw

x_float = data[0, 1:]

x_int8 = np.round(x_float * INPUT_SCALE)
x_int8 = np.clip(x_int8, -128, 127).astype(np.int8)

input_name = model.graph.input[0].name
input_shape = model.get_tensor_shape(input_name)

x = x_int8.astype(np.float32).reshape(input_shape)

print("model input name :", input_name)
print("model input shape:", input_shape)
print("label_raw:", label_raw)
print("label    :", label)
print("x_float min/max:", x_float.min(), x_float.max())
print("x_int8  min/max:", x_int8.min(), x_int8.max())
print("x_int8 nonzero:", np.count_nonzero(x_int8), "/", x_int8.size)
print("x_int8 first 20:", x_int8[:20])


# 找到 Mul_6 这个 node object
mul6_node = None
for n in model.graph.node:
    if n.name == "Mul_6":
        mul6_node = n
        break

if mul6_node is None:
    print("Cannot find Mul_6. Existing Mul nodes are:")
    for n in model.graph.node:
        if n.op_type == "Mul":
            print(n.name, list(n.input), list(n.output))
    raise RuntimeError("Cannot find Mul_6")

print("\nFound Mul_6")
print("inputs :", list(mul6_node.input))
print("outputs:", list(mul6_node.output))

# Mul_6 的第一个输入就是 final scale 之前的 raw accumulator
mul6_input = mul6_node.input[0]

# 注意：end_node 要传 node object，不是字符串
ctx = oxe.execute_onnx(
    model,
    {input_name: x},
    return_full_exec_context=True,
    end_node=mul6_node
)

raw_before_scale = ctx[mul6_input]

print("\nraw before final Mul_6:")
print("tensor name:", mul6_input)
print("shape:", raw_before_scale.shape)
print("dtype:", raw_before_scale.dtype)
print("value:", raw_before_scale.reshape(-1))

OUT_SCALE = np.float32(7.6293945e-06)
OUT_BIAS = np.array([
    -0.00854492,
     0.01147461,
     0.01098633,
     0.01196289,
    -0.01757812,
], dtype=np.float32)

raw_logits = raw_before_scale.reshape(-1).astype(np.float32)
logits_affine = raw_logits * OUT_SCALE + OUT_BIAS

print("\nlogits after Mul_6 + Add_3 manually:")
print(logits_affine)

print("\npred raw   :", int(np.argmax(raw_logits)))
print("pred affine:", int(np.argmax(logits_affine)))
print("label      :", label)

model input name : global_in
model input shape: [1, 1, 140, 1]
label_raw: 1
label    : 0
x_float min/max: -4.5744715 3.6908443
x_int8  min/max: -5 4
x_int8 nonzero: 73 / 140
x_int8 first 20: [ 4  1 -2 -4 -5 -3 -2 -1 -1  0  1  1  1  1  1  1  1  1  1  1]

Found Mul_6
inputs : ['Transpose_17_out0', 'Mul_6_param0']
outputs: ['Mul_6_out0']

raw before final Mul_6:
tensor name: Transpose_17_out0
shape: (1, 5, 1, 1)
dtype: float32
value: [ 283592. -299114. -290654. -305812. -368736.]

logits after Mul_6 + Add_3 manually:
[ 2.1550903 -2.270584  -2.2065277 -2.3211975 -2.8308105]

pred raw   : 0
pred affine: 0
label      : 0


In [2]:
from pathlib import Path
import time

paths = [
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/output_dir/bitfile/finn-accel.bit"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/output_dir/bitfile/finn-accel.hwh"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/output_dir/driver/driver.py"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_streamline.onnx"),
    Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx"),
]

for p in paths:
    print(p)
    print("  exists:", p.exists())
    if p.exists():
        print("  size :", p.stat().st_size)
        print("  mtime:", time.ctime(p.stat().st_mtime))

/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/output_dir/bitfile/finn-accel.bit
  exists: False
/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/output_dir/bitfile/finn-accel.hwh
  exists: False
/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/output_dir/driver/driver.py
  exists: False
/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_streamline.onnx
  exists: True
  size : 400774
  mtime: Thu Apr 30 11:39:03 2026
/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx
  exists: True
  size : 460056
  mtime: Thu Apr 30 11:54:29 2026


In [4]:
import os
os.environ["SANITIZE_QUANT_TENSORS"] = "0"

import numpy as np
from pathlib import Path

from qonnx.core.modelwrapper import ModelWrapper
import finn.core.onnx_exec as oxe

from finn.transformation.fpgadataflow.prepare_cppsim import PrepareCppSim
from finn.transformation.fpgadataflow.compile_cppsim import CompileCppSim
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode


BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
MODEL_PATH = BUILD_DIR / "intermediate_models" / "step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx"
TEST_TXT = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/datasets/ECG5000/ECG5000_TEST.txt")

INPUT_SCALE = 1.0

print("Using model:", MODEL_PATH)

model = ModelWrapper(str(MODEL_PATH))

model = model.transform(PrepareCppSim())
model = model.transform(CompileCppSim())
model = model.transform(SetExecMode("cppsim"))

# model-level exec_mode 必须留空，逐 node 执行
model.set_metadata_prop("exec_mode", "")

data = np.loadtxt(TEST_TXT, dtype=np.float32)

label_raw = int(data[0, 0])
label = label_raw - 1
x_float = data[0, 1:]

x_int8 = np.round(x_float * INPUT_SCALE)
x_int8 = np.clip(x_int8, -128, 127).astype(np.int8)

input_name = model.graph.input[0].name
input_shape = model.get_tensor_shape(input_name)

print("input name :", input_name)
print("input shape:", input_shape)

x = x_int8.astype(np.float32).reshape(input_shape)

ctx = oxe.execute_onnx(
    model,
    {input_name: x},
    return_full_exec_context=True
)

out_name = model.graph.output[0].name
out = ctx[out_name]

print("\nlabel:", label)
print("output name:", out_name)
print("output shape:", out.shape)
print("output dtype:", out.dtype)
print("output:", out.reshape(-1))
print("pred:", int(np.argmax(out.reshape(-1))))

# 打印最后 4480 feature 和 final output
for name in ["ConvolutionInputGenerator_6_out0", "MVAU_8_out0", "global_out"]:
    if name in ctx:
        arr = ctx[name]
        print("\n", name)
        print(" shape:", arr.shape)
        print(" min/max:", arr.min(), arr.max())
        print(" first:", arr.reshape(-1)[:20])

Using model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx


FileNotFoundError: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/data/ECG5000_TEST.txt not found.

In [5]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx")
model = ModelWrapper(str(MODEL_PATH))

print("graph outputs:")
for o in model.graph.output:
    print(o.name, model.get_tensor_shape(o.name), model.get_tensor_datatype(o.name))

nodes = list(model.graph.node)

print("\nlast 25 nodes:")
for i, n in list(enumerate(nodes))[-25:]:
    print("\n", i, n.name, n.op_type, n.domain)
    print(" inputs :", list(n.input))
    print(" outputs:", list(n.output))
    for out in n.output:
        try:
            print("  out shape:", model.get_tensor_shape(out))
            print("  out dtype:", model.get_tensor_datatype(out))
        except:
            pass

graph outputs:
global_out [1, 1, 1, 5] INT32

last 25 nodes:

 69 StreamingFIFO_rtl_36 StreamingFIFO_rtl finn.custom_op.fpgadataflow.rtl
 inputs : ['MVAU_hls_6_out0']
 outputs: ['StreamingFIFO_rtl_36_out0']
  out shape: [1, 140, 1, 32]
  out dtype: UINT8

 70 StreamingDataWidthConverter_rtl_8 StreamingDataWidthConverter_rtl finn.custom_op.fpgadataflow.rtl
 inputs : ['StreamingFIFO_rtl_36_out0']
 outputs: ['StreamingDataWidthConverter_rtl_8_out0']
  out shape: [1, 140, 1, 32]
  out dtype: FLOAT32

 71 StreamingFIFO_rtl_37 StreamingFIFO_rtl finn.custom_op.fpgadataflow.rtl
 inputs : ['StreamingDataWidthConverter_rtl_8_out0']
 outputs: ['StreamingFIFO_rtl_37_out0']
  out shape: [1, 140, 1, 32]
  out dtype: UINT8

 72 FMPadding_rtl_5 FMPadding_rtl finn.custom_op.fpgadataflow.rtl
 inputs : ['StreamingFIFO_rtl_37_out0']
 outputs: ['FMPadding_rtl_5_out0']
  out shape: [1, 148, 1, 32]
  out dtype: UINT8

 73 StreamingFIFO_rtl_38 StreamingFIFO_rtl finn.custom_op.fpgadataflow.rtl
 inputs : ['FMPa

In [2]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
IM_DIR = BUILD_DIR / "intermediate_models"

for p in sorted(IM_DIR.glob("step_*.onnx")):
    try:
        model = ModelWrapper(str(p))
        ops = [n.op_type for n in model.graph.node]
        has_cig_4480 = False

        for n in model.graph.node:
            for out in n.output:
                try:
                    shp = model.get_tensor_shape(out)
                    if shp == [1, 1, 1, 4480]:
                        has_cig_4480 = True
                except:
                    pass

        print(
            p.name,
            "nodes=", len(model.graph.node),
            "CIG=", any("ConvolutionInputGenerator" in op for op in ops),
            "has [1,1,1,4480]=", has_cig_4480,
            "MVAU_rtl=", any(op == "MVAU_rtl" for op in ops),
        )
    except Exception as e:
        print(p.name, "ERROR:", e)

step_absorb_mul_pad_mt_patched.onnx nodes= 58 CIG= False has [1,1,1,4480]= True MVAU_rtl= False
step_after_convert_to_hw_cleanup.onnx nodes= 37 CIG= True has [1,1,1,4480]= True MVAU_rtl= False
step_after_convert_to_hw_cleanup_fmpad.onnx nodes= 38 CIG= True has [1,1,1,4480]= True MVAU_rtl= False
step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx nodes= 37 CIG= True has [1,1,1,4480]= True MVAU_rtl= False
step_after_streamline_cleanup.onnx nodes= 58 CIG= False has [1,1,1,4480]= True MVAU_rtl= False
step_apply_folding_config.onnx nodes= 32 CIG= True has [1,1,1,4480]= True MVAU_rtl= True
step_convert_to_hw.onnx nodes= 68 CIG= True has [1,1,1,4480]= False MVAU_rtl= False
step_convert_to_hw_after_extra_cleanup.onnx nodes= 55 CIG= True has [1,1,1,4480]= False MVAU_rtl= False
step_convert_to_hw_patched.onnx nodes= 56 CIG= True has [1,1,1,4480]= True MVAU_rtl= False
step_create_dataflow_partition.onnx nodes= 1 CIG= False has [1,1,1,4480]= False MVAU_rtl= False
step_create_dataflow_partition_fo

In [3]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx")
model = ModelWrapper(str(MODEL_PATH))

for i, n in enumerate(model.graph.node):
    if n.name == "MVAU_rtl_0":
        print("index:", i)
        print("node :", n.name, n.op_type)
        print("inputs :", list(n.input))
        print("outputs:", list(n.output))

        print("\nPrevious nodes:")
        for j in range(max(0, i - 8), i + 2):
            m = model.graph.node[j]
            print("\n", j, m.name, m.op_type)
            print(" inputs :", list(m.input))
            print(" outputs:", list(m.output))
            for out in m.output:
                try:
                    print("  ", out, model.get_tensor_shape(out), model.get_tensor_datatype(out))
                except:
                    pass

index: 92
node : MVAU_rtl_0 MVAU_rtl
inputs : ['StreamingFIFO_rtl_47_out0', 'MVAU_rtl_0_param0']
outputs: ['MVAU_rtl_0_out0']

Previous nodes:

 84 Thresholding_rtl_3 Thresholding_rtl
 inputs : ['StreamingFIFO_rtl_43_out0', 'Thresholding_rtl_3_param0']
 outputs: ['Thresholding_rtl_3_out0']
   Thresholding_rtl_3_out0 [1, 140, 1, 32] UINT8

 85 StreamingFIFO_rtl_44 StreamingFIFO_rtl
 inputs : ['Thresholding_rtl_3_out0']
 outputs: ['StreamingFIFO_rtl_44_out0']
   StreamingFIFO_rtl_44_out0 [1, 140, 1, 32] UINT8

 86 StreamingDataWidthConverter_rtl_11 StreamingDataWidthConverter_rtl
 inputs : ['StreamingFIFO_rtl_44_out0']
 outputs: ['StreamingDataWidthConverter_rtl_11_out0']
   StreamingDataWidthConverter_rtl_11_out0 [1, 140, 1, 32] FLOAT32

 87 StreamingFIFO_rtl_45 StreamingFIFO_rtl
 inputs : ['StreamingDataWidthConverter_rtl_11_out0']
 outputs: ['StreamingFIFO_rtl_45_out0']
   StreamingFIFO_rtl_45_out0 [1, 140, 1, 32] UINT8

 88 ConvolutionInputGenerator_rtl_6 ConvolutionInputGenerator_rt

In [1]:
import json
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
IM_DIR = BUILD_DIR / "intermediate_models"

# 改成你的实际 folding config 路径
FOLDING_CONFIG = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/manual_folding_config.json")

# folding config 应该对 apply_folding_config 之前的模型生效
MODEL_BEFORE_FOLD = IM_DIR / "step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx"

model = ModelWrapper(str(MODEL_BEFORE_FOLD))

node_names = set(n.name for n in model.graph.node)

with open(FOLDING_CONFIG, "r") as f:
    cfg = json.load(f)

cfg_names = set(cfg.keys())

matched = sorted(cfg_names & node_names)
missing = sorted(cfg_names - node_names)
not_configured = sorted(node_names - cfg_names)

print("model:", MODEL_BEFORE_FOLD)
print("folding config:", FOLDING_CONFIG)

print("\nMatched config entries:", len(matched))
for n in matched:
    print("  MATCH:", n)

print("\nConfig entries NOT found in model:", len(missing))
for n in missing:
    print("  MISSING:", n)

print("\nModel nodes without config:", len(not_configured))
for n in not_configured:
    print("  NO_CONFIG:", n)

model: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx
folding config: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/manual_folding_config.json

Matched config entries: 0

Config entries NOT found in model: 10
  MISSING: Defaults
  MISSING: MVAU_hls_0
  MISSING: MVAU_hls_1
  MISSING: MVAU_hls_2
  MISSING: MVAU_hls_3
  MISSING: MVAU_hls_4
  MISSING: MVAU_hls_5
  MISSING: MVAU_hls_6
  MISSING: MVAU_hls_7
  MISSING: MVAU_rtl_0

Model nodes without config: 37
  NO_CONFIG: AddStreams_0
  NO_CONFIG: AddStreams_1
  NO_CONFIG: AddStreams_2
  NO_CONFIG: Add_0
  NO_CONFIG: ConvolutionInputGenerator_0
  NO_CONFIG: ConvolutionInputGenerator_1
  NO_CONFIG: ConvolutionInputGenerator_2
  NO_CONFIG: ConvolutionInputGenerator_3
  NO_CONFIG: ConvolutionInputGenerator_4
  NO_CONFIG: ConvolutionInputGenerator_5
  NO_CONFIG: ConvolutionInputGenerator_6
  NO_CONFIG: DuplicateStreams_0

In [3]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper

MODEL_PATH = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_set_fifo_depths.onnx")

model = ModelWrapper(str(MODEL_PATH))

for n in model.graph.node:
    if "MVAU" in n.op_type:
        print(n.name, n.op_type)
        print(" inputs :", list(n.input))
        print(" outputs:", list(n.output))

MVAU_hls_0 MVAU_hls
 inputs : ['StreamingFIFO_rtl_2_out0', 'MVAU_hls_0_param0', 'MVAU_hls_0_param1']
 outputs: ['MVAU_hls_0_out0']
MVAU_hls_1 MVAU_hls
 inputs : ['StreamingFIFO_rtl_5_out0', 'MVAU_hls_1_param0', 'MVAU_hls_1_param1']
 outputs: ['MVAU_hls_1_out0']
MVAU_hls_2 MVAU_hls
 inputs : ['StreamingFIFO_rtl_10_out0', 'MVAU_hls_2_param0', 'MVAU_hls_2_param1']
 outputs: ['MVAU_hls_2_out0']
MVAU_hls_3 MVAU_hls
 inputs : ['StreamingFIFO_rtl_20_out0', 'MVAU_hls_3_param0', 'MVAU_hls_3_param1']
 outputs: ['MVAU_hls_3_out0']
MVAU_hls_4 MVAU_hls
 inputs : ['StreamingFIFO_rtl_25_out0', 'MVAU_hls_4_param0', 'MVAU_hls_4_param1']
 outputs: ['MVAU_hls_4_out0']
MVAU_hls_5 MVAU_hls
 inputs : ['StreamingFIFO_rtl_29_out0', 'MVAU_hls_5_param0', 'MVAU_hls_5_param1']
 outputs: ['MVAU_hls_5_out0']
MVAU_hls_6 MVAU_hls
 inputs : ['StreamingFIFO_rtl_35_out0', 'MVAU_hls_6_param0', 'MVAU_hls_6_param1']
 outputs: ['MVAU_hls_6_out0']
MVAU_hls_7 MVAU_hls
 inputs : ['StreamingFIFO_rtl_40_out0', 'MVAU_hls_7_param0

In [4]:
from pathlib import Path
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp

BUILD_DIR = Path("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2")
IM_DIR = BUILD_DIR / "intermediate_models"

MODEL_IN = IM_DIR / "step_target_fps_parallelization.onnx"
MODEL_OUT = IM_DIR / "step_target_fps_parallelization_force_final_hls.onnx"

model = ModelWrapper(str(MODEL_IN))

for n in model.graph.node:
    if n.name == "MVAU_8":
        print("Patching final MVAU:", n.name)
        op = getCustomOp(n)

        # 保守设置
        op.set_nodeattr("PE", 1)
        op.set_nodeattr("SIMD", 1)

        # 关键：避免 specialize 成 MVAU_rtl
        op.set_nodeattr("mem_mode", "internal_embedded")

        try:
            op.set_nodeattr("ram_style", "auto")
        except Exception:
            pass

model.save(str(MODEL_OUT))
print("saved:", MODEL_OUT)

saved: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_target_fps_parallelization_force_final_hls.onnx
